This notebook will process the output BCR cell annotations csv from Cellismo, which has the IGHV annotations, and define the subsets. 

The data should have the clones, non-B, multiples, undetermined and unassigned removed, and be annotated with my groups before processing in this. 
Ther annotation above will be done by the AbSeq stage, so that cellismo annotation can be used. 

## Set-up ##

In [ ]:
# Import python modules

import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir 
import muon as mu
import skbio as sk
import os as os
import anndata as adata
from muon import MuData
from matplotlib import pyplot as plt, cm as mpl_cm
from cycler import cycler

#sc.set_figure_params(figsize=(6, 6))  ## generates an error, get this code for current versions if its needed.
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [ ]:
sc.logging.print_header()

# Set-up paths etc

In [ ]:
# Change paths throughout notebook
# Data
work_folder = "PATH/runFolders/AllData"
VDJperCell_file = "BCR_CLLremoved_AbSeq_CellAnnotations.csv"
VAR = "GroupA_arm"
ARM = "Ctl"
out_folder = "PATH/stats/outputs_BCRstereotypy_arrest"

In [ ]:
print (out_folder)

## Get the data in ##

In [ ]:
vdj = pd.read_csv(work_folder + "/" + VDJperCell_file) 

In [ ]:
vdj

In [ ]:
vdj.shape

# For batched data I will need to check if columns are same or different for cell id and batch. If they are different, I will need to concatenate these to make a unique identifier per batch. 

# Select the columns I want

In [ ]:
vdj_filtered = vdj[["Cell_Index",
                     VAR, 
                     "BCR_Heavy_CDR3_Junction_Nucleotide_Dominant", 
                     "BCR_Heavy_CDR3_Junction_Translation_Dominant", 
                     "BCR_Heavy_V_gene_Dominant", 
                     "BCR_Heavy_D_gene_Dominant", 
                     "BCR_Heavy_J_gene_Dominant", 
                     "BCR_Light_CDR3_Junction_Nucleotide_Dominant", 
                     "BCR_Light_CDR3_Junction_Translation_Dominant", 
                     "BCR_Light_V_gene_Dominant",
                     "BCR_Light_J_gene_Dominant" 
                     ]]

Export the object as a .csv for Arrest subset analysis

In [ ]:
FILE= os.path.join(out_folder, "VDJ_forArrest_groupA_plusLCs.csv")

vdj_filtered.to_csv(path_or_buf= FILE)

In [ ]:
vdj_filtered

In [ ]:
vdj_filtered["VDJ_calls"] = vdj_filtered[["BCR_Heavy_V_gene_Dominant", "BCR_Heavy_D_gene_Dominant", "BCR_Heavy_J_gene_Dominant"]].agg(''.join, axis=1)

In [ ]:
vdj_filtered

# Select a category to investigate e.g. sample name or group arm. 

In [ ]:
# Make a dataframe of the subset
ArmSample = vdj_filtered[vdj_filtered[VAR].isin([ARM])]

In [ ]:
ArmSample

# Get an abundance of each VDJ_calls

In [ ]:
vdj_counts = ArmSample.value_counts("VDJ_calls", sort=True, dropna=True)
vdj_counts

In [ ]:
vdj_counts_df = []
for k, v, in vdj_counts.items():
    print(k, v)
    VDJ_calls = k
    vdj_counts_df.append(dict(VDJ_calls=VDJ_calls, Count=v))
vdj_counts_df1 = pd.DataFrame(vdj_counts_df)
vdj_counts_df1.head()

In [ ]:
vdj_counts_df1.shape

In [ ]:
# Make this dataframe a csv

FILE= os.path.join(out_folder, (ARM + "_VDJ_freq.csv"))

vdj_counts_df1.to_csv(path_or_buf= FILE)

# Make a pretty plot

In [ ]:
# Create a subset of the data first, for example the top 10 VDJs

vdj_topTen = vdj_counts_df1.head(10)

In [ ]:
NAME = VAR + ARM + "_vdjBar.png"
FILE= os.path.join(out_folder, NAME)

vdj_topTen.plot.bar(
    x="VDJ_calls", 
    y="Count", 
    color = ["blue", "red"], 
    xlabel= "VDJ", 
    ylabel = "Frequency", 
    rot = 90,
    title = "Top 10 VDJ Frequences in PostRx Cases")

plt.savefig(
    fname=FILE,
    bbox_inches="tight"
)

In [ ]:
vdj_topTen.plot.pie(
    y= "Count",
    ylabel = "VDJ_calls")